In [1]:
# Step 1: Import libraries
import pandas as pd
import numpy as np

# Export to DataBase
from sqlalchemy import create_engine, types

#  Physical activity data from Questionnaire

Data file on Physical activity  were downloaded from - https://wwwn.cdc.gov/nchs/nhanes/search/datapage.aspx?Component=Questionnaire&Cycle=2021-2023

In [2]:
# Exporting the Dietary Nutrients intake day  data
df_PA = pd.read_sas('../data/raw_data/PAQ_L.xpt', format="xport", encoding="utf-8")

In [3]:
from dotenv import dotenv_values

config = dotenv_values()

# define variables for the login
pg_user = config['POSTGRES_USER']  # align the key label with your .env file !
pg_host = config['POSTGRES_HOST']
pg_port = config['POSTGRES_PORT']
pg_db = config['POSTGRES_DB']
pg_schema = config['POSTGRES_SCHEMA']
pg_pass = config['POSTGRES_PASS']

# Now building the URL with the values from the .env file
url = f'postgresql://{pg_user}:{pg_pass}@{pg_host}:{pg_port}/{pg_db}'

engine = create_engine(url, echo=False) 

In [4]:
# Step 1: Rename the columns

df_PA = df_PA.rename(columns={
    'SEQN': 'seqn_no',
    'PAD790Q': 'mod_LTPA_count', # moderate LTPA - leisure time physical activity
    'PAD790U': 'mod_LTPA_unit',
    'PAD800': 'mod_LTP_minutes',
    'PAD810Q': 'vig_LTP_count', # vigorous LTPA
    'PAD810U': 'vig_LTP_unit',
    'PAD820': 'vig_LTP_minutes',
    'PAD680': 'sed_minutes' # Sedantary activity minutes excluding sleep
})

# Step 2: Changing the data type of 'seqn_no' to int
df_PA['seqn_no'] = df_PA['seqn_no'].astype(int)

In [5]:
df_PA.columns

Index(['seqn_no', 'mod_LTPA_count', 'mod_LTPA_unit', 'mod_LTP_minutes',
       'vig_LTP_count', 'vig_LTP_unit', 'vig_LTP_minutes', 'sed_minutes'],
      dtype='object')

In [6]:
# Fill NaN values in activity minutes and counts with 0, and units with an empty string for easier processing

df_PA['mod_LTP_minutes'] = df_PA['mod_LTP_minutes'].fillna(0)
df_PA['mod_LTPA_unit'] = df_PA['mod_LTPA_unit'].fillna('')
df_PA['vig_LTP_minutes'] = df_PA['vig_LTP_minutes'].fillna(0)
df_PA['vig_LTP_unit'] = df_PA['vig_LTP_unit'].fillna('')
df_PA['mod_LTPA_count'] = df_PA['mod_LTPA_count'].fillna(0)
df_PA['vig_LTP_count'] = df_PA['vig_LTP_count'].fillna(0)

# Initialize new columns for weekly minutes
df_PA['moderate_activity_minutes_per_week'] = 0.0
df_PA['vigorous_activity_minutes_per_week'] = 0.0

# Function to convert minutes based on unit to weekly minutes
def convert_to_weekly(row, minutes_col, unit_col, count_col):
    unit = str(row[unit_col]).strip().upper()
    minutes = row[minutes_col]
    count = row[count_col]

    if unit == 'W':
        return minutes * count
    elif unit == 'D':
        return minutes * count * 7
    elif unit == 'M':
        # Approximate conversion for monthly to weekly (assuming 4 weeks in a month)
        return minutes * count / 4
    else:
        return 0.0

# Apply the conversion function
df_PA['moderate_activity_minutes_per_week'] = df_PA.apply(lambda row: convert_to_weekly(row, 'mod_LTP_minutes', 'mod_LTPA_unit', 'mod_LTPA_count'), axis=1)
df_PA['vigorous_activity_minutes_per_week'] = df_PA.apply(lambda row: convert_to_weekly(row, 'vig_LTP_minutes', 'vig_LTP_unit', 'vig_LTP_count'), axis=1)

# Define activity classification function based on WHO guidelines for adults
# WHO recommends:
# At least 150–300 minutes of moderate-intensity aerobic physical activity OR
# At least 75–150 minutes of vigorous-intensity aerobic physical activity OR
# An equivalent combination across the week
# One minute of vigorous-intensity activity is equivalent to 2 minutes of moderate-intensity activity.

def classify_activity_level(row):
    moderate_minutes = row['moderate_activity_minutes_per_week']
    vigorous_minutes = row['vigorous_activity_minutes_per_week']

    # Calculate MET-minutes equivalent
    # 1 minute of vigorous activity is equivalent to 2 minutes of moderate activity
    equivalent_moderate_minutes = moderate_minutes + (vigorous_minutes * 2)

    # Classification based on WHO guidelines
    if equivalent_moderate_minutes >= 150:
        return 'Active'
    else:
        return 'Not Active'

# Apply the classification function
df_PA['activity_level'] = df_PA.apply(classify_activity_level, axis=1)

# Display a sample of the results with the new columns
print(df_PA[['moderate_activity_minutes_per_week', 'vigorous_activity_minutes_per_week', 'activity_level', 'sed_minutes']].head(10))

# Display the value counts for activity_level
print("\nActivity Level Distribution:")
print(df_PA['activity_level'].value_counts())

   moderate_activity_minutes_per_week  vigorous_activity_minutes_per_week  \
0                               135.0                               135.0   
1                               180.0                               135.0   
2                                20.0                                 0.0   
3                                 0.0                                 0.0   
4                               630.0                                60.0   
5                                30.0                                 7.5   
6                                 0.0                                 0.0   
7                               900.0                                 0.0   
8                               135.0                                15.0   
9                                60.0                                 0.0   

  activity_level  sed_minutes  
0         Active        360.0  
1         Active        480.0  
2     Not Active        240.0  
3     Not Active        

In [7]:
#export Physical Activity as csv to data/analysis_data for further analysis

file_path = "../data/analysis_data/Physical_activity.csv" 

try:
    df_PA.to_csv(file_path, index=False, encoding='utf-8')
    print(f"DataFrame successfully saved to: {file_path}")
except Exception as e:
    print(f"Error saving DataFrame to CSV: {e}")

DataFrame successfully saved to: ../data/analysis_data/Physical_activity.csv


In [8]:
# creating table and adding avg food data to the database
df_PA.to_sql(name = 'questionnaire_physical_activity', con=engine, schema='capstone_group_3',if_exists='replace',index=False)

153

Descriptive statistics

In [9]:
# Describe sedentary minutes
print("Descriptive statistics for Sedentary Minutes:")
print(df_PA['sed_minutes'].describe())

# Analyze sedentary minutes by activity level
print("\nAverage Sedentary Minutes by Activity Level:")
print(df_PA.groupby('activity_level')['sed_minutes'].mean())


Descriptive statistics for Sedentary Minutes:
count    8.138000e+03
mean     4.469827e+02
std      9.174642e+02
min      5.397605e-79
25%      1.800000e+02
50%      3.000000e+02
75%      4.800000e+02
max      9.999000e+03
Name: sed_minutes, dtype: float64

Average Sedentary Minutes by Activity Level:
activity_level
Active        389.964750
Not Active    511.243335
Name: sed_minutes, dtype: float64


Average Sedentary Minutes by Activity Level:

As expected, there's a difference in sedentary minutes between the two activity groups:

Active individuals: Average sedentary minutes per day is about 390 minutes (6.5 hours).
Not Active individuals: Average sedentary minutes per day is about 511 minutes (8.5 hours).

# Sleep disorder data from Questionnaire

In [10]:
# Exporting sleep disorder data from Questionaire
df_quest_sleep = pd.read_sas('../data/raw_data/SLQ_L.xpt', format="xport", encoding="utf-8")

In [11]:
# Step 1: Rename the columns

df_quest_sleep = df_quest_sleep.rename(columns={
    'SEQN': 'seqn_no',
    'SLQ300': 'bedtime_weekday',
    'SLQ310': 'waketime_weekday',
    'SLD012': 'sleep_hrs_weekday', 
    'SLQ320': 'bedtime_weekend',
    'SLQ330': 'waketime_weekend',
    'SLD013': 'sleep_hrs_weekend'
})

# Step 2: Changing the data type of 'seqn_no' to int
df_quest_sleep['seqn_no'] = df_quest_sleep['seqn_no'].astype(int)


In [12]:
# Equal Weight Average
df_quest_sleep['avg_sleep_hrs'] = df_quest_sleep[['sleep_hrs_weekday', 'sleep_hrs_weekend']].mean(axis=1)
df_quest_sleep

,seqn_no,bedtime_weekday,waketime_weekday,sleep_hrs_weekday,bedtime_weekend,waketime_weekend,sleep_hrs_weekend,avg_sleep_hrs
0,130378,21:30,07:00,9.5,00:00,09:00,9.0,9.25
1,130379,21:00,06:00,9.0,21:00,06:00,9.0,9.00
2,130380,00:00,08:00,8.0,00:00,09:00,9.0,8.50
3,130384,21:30,05:00,7.5,23:00,07:00,8.0,7.75
4,130385,22:05,06:15,8.0,22:05,06:15,8.0,8.00
...,...,...,...,...,...,...,...,...
8496,142305,00:00,09:00,9.0,00:00,09:00,9.0,9.00
8497,142307,00:00,07:00,7.0,00:00,07:00,7.0,7.00
8498,142308,21:30,06:30,9.0,00:00,08:00,8.0,8.50
8499,142309,21:00,05:00,8.0,00:00,12:00,12.0,10.00


In [13]:
#export Sleep disorder data as csv to data/analysis_data for further analysis

file_path = "../data/analysis_data/Sleep_disorder.csv" 

try:
    df_quest_sleep.to_csv(file_path, index=False, encoding='utf-8')
    print(f"DataFrame successfully saved to: {file_path}")
except Exception as e:
    print(f"Error saving DataFrame to CSV: {e}")

DataFrame successfully saved to: ../data/analysis_data/Sleep_disorder.csv


In [ ]:
# Now building the URL with the values from the .env file
url = f'postgresql://{pg_user}:{pg_pass}@{pg_host}:{pg_port}/{pg_db}'

engine = create_engine(url, echo=False) 

df_quest_sleep.to_sql(name = 'questionnaire_sleep', con=engine, schema='capstone_group_3',if_exists='replace',index=False)

501